# 09 - Comparison against three models from the literature survey

This notebook does two things at once, because they are the same job:

1. **Re-runs the three baselines on the correct protocol.** The review's most
   serious comment was that `mentalroberta` silently ran as `roberta-base` for
   two of three seeds, and that the baselines used batch 16 / 8 epochs / length
   256 against the proposed model's batch 32 / length 192. They also lacked
   **ordinal class weights**, which the proposed model has - and unweighted
   cross-entropy collapses the severity and risk heads onto the majority class.
   Any severity or risk margin measured against those baselines was measuring
   the class weighting, not the architecture.
2. **Adds three comparison models from your own survey**, under that same
   protocol.

Everything here shares one `TrainConfig`, the same leakage-controlled splits,
the same masked multi-task loss, the same symptom `pos_weight`, the same ordinal
class weights, the same validation-tuned thresholds, the same selection rule and
the same three seeds. Only the representation differs.

## The three models

| name | after | contests |
|---|---|---|
| `sota_ens_stack` | **[11]** Suksam, Kaewbuadee & Jareanpon (2026), *Information* 17(6), 546 | N1 - symptoms first, via a transformer **stacking ensemble** rather than a concept bottleneck |
| `sota_postgraph` | **[18]** Rehman et al. (2026), *Healthcare* 14(11), 1440 | N2 - a graph from **post semantics + author relations** rather than a DSM-5 prior |
| `sota_prompt_ssl` | **[21]** Hsieh et al. (2025), *Scientific Reports* | the multi-label setup itself, via **prompt perspectives + perturbation consistency** |

## Read this before you report any of it

**These are re-implementations of the architecture each paper describes, on this
project's label space, under this project's protocol. They are not reproductions
of the published systems, and their numbers must never be placed beside the
numbers printed in those papers.**

- The full method sections were not reachable (MDPI blocks automated access; no
  author code was found). Each architecture was built from the published
  description of its modules. Every underdetermined choice is marked
  `DEVIATION` in the class docstring in `src/symptograph/sota.py` - read them
  before writing the comparison section.
- All three papers target different label spaces on different corpora. Mapping
  them onto 22 symptoms + 4-level severity + 5-level risk + binary stress means
  giving them this project's `SharedHeads`. That isolates the representation,
  which is the point, but it also means these are not the models that produced
  the published scores.
- `[11]`'s Phase 2 (DSM-5 screening over a 14-day window) **cannot run on this
  data** - the gold split has a median of one post per author. `[18]`'s SHAP and
  LIME analyses are post-hoc and change no metric here.

**[10] Bao, Pérez & Parapar (2024) was considered and excluded.** Its
contribution is the quality of natural-language symptom explanations; stripped
of generation it reduces to the `mentalroberta` baseline already present, and
explanation quality cannot be scored by this metric suite. Reporting it as a
classification competitor would misrepresent the paper. Say this in the paper
rather than quietly omitting it.

## 1. Setup

In [ ]:
import os, sys, json, warnings
import numpy as np, pandas as pd
from google.colab import drive
warnings.filterwarnings('ignore')
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/hari/SymptoGraph-LLM'
PROCESSED  = f'{DRIVE_BASE}/processed'
RESULTS    = f'{DRIVE_BASE}/comparison'        # baselines + SOTA, one protocol
PROPOSED   = f'{DRIVE_BASE}/proposed_v2_e8'    # the 8-epoch proposed run
PROPOSED_FALLBACK = f'{DRIVE_BASE}/proposed_v2'

# Which to train this session. Costs differ a lot - see the budget cell.
MODELS = None      # None = all six

os.makedirs(RESULTS, exist_ok=True)
assert os.path.isdir(PROCESSED), f'missing {PROCESSED}'
print('writing to', RESULTS)

In [ ]:
REPO, CHECKOUT = 'https://github.com/sid-2k6/SymptoGraph-LLM.git', '/content/SymptoGraph-LLM'
if os.path.isdir(f'{CHECKOUT}/.git'):
    !git -C $CHECKOUT fetch -q origin && git -C $CHECKOUT reset -q --hard origin/main
else:
    !git clone -q --branch main $REPO $CHECKOUT
sys.path.insert(0, f'{CHECKOUT}/src')
from symptograph import baselines as B
from symptograph import sota as S          # registers the three models on import
from symptograph.preprocess import Ontology
from symptograph.metrics import headline_table, set_publication_style
from symptograph.jsonio import read_json
assert set(S.SOTA_MODELS) <= set(B.MODEL_REGISTRY), 'stale checkout'
!git -C $CHECKOUT log --oneline -1

import torch
assert torch.cuda.is_available(), 'need a GPU runtime'
print(torch.cuda.get_device_name(0),
      f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
onto = Ontology.load(f'{CHECKOUT}/configs/symptom_ontology.json')
set_publication_style(font_size=20)
for m, src in S.SOTA_SOURCES.items():
    print(f'  {m:18s} {src}')

## 2. One protocol for everything

Every field here matches `proposed_v2_e8`'s `ProposedConfig`. Cell 3 asserts it
rather than trusting the comment.

In [ ]:
ALL_MODELS = list(B.BASELINES) + list(S.SOTA_MODELS)
TRAIN = ALL_MODELS if MODELS is None else MODELS

cfg = B.TrainConfig(
    epochs=8, seeds=(0, 1, 2),
    batch_size=32, eval_batch_size=64, max_length=192,
    transformer_name='mental/mental-roberta-base',
    # the review's most serious finding: a 403 silently substituted roberta-base
    # while the run was still labelled "mentalroberta"
    strict_encoder=True,
    scheduler='linear',
    selection_metric='neg_val_loss',
    early_stopping_patience=2,
    label_smoothing=0.05,
    ordinal_class_weights=True,     # the baselines did not have this before
    pos_weight_cap=10.0,
    tune_thresholds=True,
    # the baselines never saw the ~50k weak labels. Leave this False for the
    # architecture comparison; section 7 explains the matched-data run you also
    # need before claiming the architecture alone wins.
    use_weak_labels=False)

print(f'epochs {cfg.epochs} | seeds {cfg.seeds} | batch {cfg.batch_size} | '
      f'max_length {cfg.max_length}')
print(f'selection {cfg.selection_metric} | patience {cfg.early_stopping_patience} | '
      f'label smoothing {cfg.label_smoothing}')
print(f'ordinal class weights {cfg.ordinal_class_weights} | '
      f'pos_weight cap {cfg.pos_weight_cap} | thresholds {cfg.tune_thresholds}')
print(f'strict encoder {cfg.strict_encoder}')
print('\ntraining:', TRAIN)
print('learning rates:', {m: cfg.lr_for(m) for m in TRAIN})

## 3. Protocol parity check against the proposed run

Reads the proposed model's stored config and compares every shared field. If
this fails, stop - the comparison would not be measuring architecture.

In [ ]:
prop_root = PROPOSED if os.path.isdir(PROPOSED) else PROPOSED_FALLBACK
cand = [f'{prop_root}/{r}/seed_0/config.json'
        for r in ('M4_denoised', 'M1_bottleneck', 'M0_flat')]
pc = next((read_json(c) for c in cand if os.path.isfile(c)), None)
assert pc, f'no proposed config found under {prop_root} - run notebook 08 first'
print('comparing against', prop_root)

SHARED = ['epochs', 'batch_size', 'eval_batch_size', 'max_length', 'weight_decay',
          'warmup_ratio', 'grad_clip', 'dropout', 'scheduler', 'selection_metric',
          'early_stopping_patience', 'label_smoothing', 'ordinal_class_weights',
          'pos_weight_cap', 'tune_thresholds']
rows = []
for f in SHARED:
    a, b = pc.get(f, '(absent)'), getattr(cfg, f, '(absent)')
    rows.append({'field': f, 'proposed': a, 'this_run': b, 'match': a == b})
chk = pd.DataFrame(rows)
print(chk.to_string(index=False))
print()
print('encoder  proposed:', pc.get('encoder_name'), '| here:', cfg.transformer_name)
print('seeds    proposed:', pc.get('seeds'), '| here:', list(cfg.seeds))

bad = chk[~chk.match]
if len(bad):
    print('\nMISMATCHED:', list(bad.field))
    print('Fix these before training, or the comparison measures protocol, '
          'not architecture.')
else:
    print('\nprotocol parity: OK')

## 4. Budget

Per-model cost on an L4, 3 seeds, up to 8 epochs with early stopping. The two
multi-pass models are the expensive ones.

| model | passes per step | rough cost |
|---|---|---|
| `tfidf_mlp` | - | a few minutes |
| `bilstm_attn` | - | a few minutes |
| `mentalroberta` | 1 | ~30 min |
| `sota_postgraph` | 1 + batch graph | ~35 min |
| `sota_prompt_ssl` | 2 prompt views + 1 fused | ~70 min |
| `sota_ens_stack` | 2 encoders, gradient checkpointed | ~80 min |

**About 3.5 h for all six.** Set `MODELS` in cell 1 to split it across sessions -
everything resumes. If you are short, the ordering that preserves the most
meaning is: `mentalroberta` first (it fixes the broken baseline), then
`sota_postgraph`, then the other two.

## 5. Train

In [ ]:
results = {}
for name in TRAIN:
    print('=' * 72)
    print(name, '|', S.SOTA_SOURCES.get(name, 'baseline'))
    print('=' * 72)
    results[name] = B.run_baseline(name, cfg, PROCESSED, RESULTS,
                                   onto=onto, resume=True, verbose=True)
print('\ntrained:', list(results))

## 6. Integrity checks

Run before any metric. The first one is the review comment that mattered most.

In [ ]:
rows = []
for name in ALL_MODELS:
    for s in cfg.seeds:
        p = f'{RESULTS}/{name}/seed_{s}/performance_metrics.json'
        if not os.path.isfile(p):
            continue
        pm = read_json(p)
        rows.append({'model': name, 'seed': s, 'encoder': pm.get('encoder'),
                     'fell_back': pm.get('encoder_fallback_used'),
                     'lr': pm.get('learning_rate'),
                     'best_epoch': pm.get('best_epoch'),
                     'epochs': pm.get('epochs_completed')})
enc = pd.DataFrame(rows)
print(enc.to_string(index=False))
fell = enc[enc.fell_back == True]
print()
if len(fell):
    print('ENCODER FALLBACK in:', list(zip(fell.model, fell.seed)))
    print('Those runs are NOT MentalRoBERTa. Delete and re-run them.')
else:
    print('no encoder fell back - every transformer run used',
          cfg.transformer_name)

print('\nper-example predictions present (needed for the paired tests):')
for name in ALL_MODELS:
    have = [s for s in cfg.seeds
            if os.path.isfile(f'{RESULTS}/{name}/seed_{s}/predictions_test.npz')]
    if os.path.isdir(f'{RESULTS}/{name}'):
        print(f'  {name:18s} seeds {have}')

## 7. The comparison table

In [ ]:
from symptograph.metrics import compare_runs

runs = {n: f'{RESULTS}/{n}' for n in ALL_MODELS
        if os.path.isfile(f'{RESULTS}/{n}/aggregate/aggregate_metrics.json')}
# the proposed model, from whichever run is current
for rung in ('M4_denoised', 'M1_bottleneck'):
    f = f'{prop_root}/{rung}/aggregate/aggregate_metrics.json'
    if os.path.isfile(f):
        runs[f'SymptoGraph-v2 ({rung})'] = f'{prop_root}/{rung}'

tabs = []
for label, path in runs.items():
    tabs.append(headline_table(
        read_json(f'{path}/aggregate/aggregate_metrics.json')['aggregate'],
        label=label))
tt = pd.concat(tabs, ignore_index=True)
print(tt.pivot(index='metric', columns='model', values='formatted').to_string())
tt.to_csv(f'{RESULTS}/comparison_table.csv', index=False)

print('\nlower is better: severity_depression_mae, severity_risk_mae')
print('note: the baselines and comparison models saw NO weak labels, while the')
print('proposed M3/M4 saw ~50k. Part of any margin is extra supervision. For a')
print('matched-data comparison re-run this with use_weak_labels=True.')

In [ ]:
# held-out test split, from each seed's best-validation checkpoint
rows = []
for label, path in runs.items():
    f = f'{path}/aggregate/aggregate_metrics.json'
    if os.path.isfile(f):
        by = read_json(f).get('aggregate_by_split', {})
        node = by.get('test')
        if node:
            rows.append(headline_table(node, label=label))
if rows:
    tc = pd.concat(rows, ignore_index=True)
    print(tc.pivot(index='metric', columns='model', values='formatted').to_string())
    tc.to_csv(f'{RESULTS}/test_comparison.csv', index=False)
else:
    print('no per-split aggregates found; the validation table above is what you have')

## 8. Paired significance tests

The proposed model against each comparison model, resampling the **same** test
examples on both sides. A difference counts only if it clears p < 0.05 on every
seed **and** keeps the same sign on every seed.

Remember the multiple-comparison problem: 5 metrics x 6 models is 30 tests, so
about 1.5 false positives are expected at alpha = 0.05. Bonferroni gives
alpha = 0.0017.

In [ ]:
from symptograph.significance import compare_models

prop_rung = ('M4_denoised'
             if os.path.isdir(f'{prop_root}/M4_denoised') else 'M1_bottleneck')
out = []
for name in ALL_MODELS:
    if not os.path.isdir(f'{RESULTS}/{name}'):
        continue
    for row in compare_models(f'{RESULTS}/{name}', f'{prop_root}/{prop_rung}',
                              seeds=cfg.seeds, split='test', n_boot=1000,
                              label_a=name, label_b='proposed'):
        row['comparison'] = f'{name} -> proposed'
        out.append(row)
sig = pd.DataFrame(out)
keep = ['comparison', 'metric', 'max_p', 'consistent_sign', 'significant',
        'delta_per_seed']
print(sig[[c for c in keep if c in sig.columns]].to_string(index=False))
sig.to_csv(f'{RESULTS}/paired_vs_proposed.csv', index=False)
n = int(sig.significant.sum()) if 'significant' in sig else 0
print(f'\n{n} of {len(sig)} comparisons significant and sign-consistent.')
print(f'Expected by chance at alpha=0.05: about {0.05*len(sig):.1f}.')

## 9. What you can and cannot claim

**Can claim**, if the table supports it: that the proposed model matches or
exceeds a correctly-trained MentalRoBERTa baseline and three architectures from
the 2024-2026 literature, **under one identical protocol with leakage-controlled
splits**, with paired significance tests reported either way. The protocol
parity check in section 3 and the encoder check in section 6 are what make that
claim auditable.

**Cannot claim** that you reproduced or beat the published results of [11], [18]
or [21]. You re-implemented their described architectures on a different label
space; the `DEVIATION` notes in `src/symptograph/sota.py` list exactly where the
implementations are underdetermined. Write that paragraph explicitly - a
reviewer who knows those papers will look for it, and its absence is what turns
a fair comparison into an overclaim.

**Still outstanding:**
1. A **matched-data** run with `use_weak_labels=True`, so the ~50k weak labels
   are not doing work the architecture gets credit for.
2. The in-house anxiety gold set - the 11 weak-only symptoms still have no gold
   labels anywhere, so `symptom_macro_auroc[weak_only]` is still a dash for
   every model in this table.
3. Clinician review of the symptom-disorder prior.